# v1w · v1 rebuilt with weekly decisions

Plan: [docs/weekly_rebuild_note.md](../docs/weekly_rebuild_note.md), fixed and hashed before this notebook existed.

**Unchanged from v1:**
- the data, ±1% close→close label, features, candidates and tuning;
- **model selection by validation macro-F1**, which doesn't depend on the trading schedule, so the random forest E06 stays;
- the refit on Train + Val, and cash earning 0% as v1 was built.

v1's saved validation probabilities (model trained on 2010–2020) and test probabilities (refit on 2010–2022) are used as they are.

**Re-selected on a weekly schedule:** the signal rule. The margin is chosen from {argmax, 0, 0.05, 0.10, 0.20} by the highest after-cost Sharpe on the **2021–22 weekly** validation backtest, with ties going to fewer trades. Each Friday after the close, the rule turns the model's probabilities into Buy, Hold or Sell for the next 5 trading days. The trade fills at Monday's open, and the position is held all week.

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from src import config, v2, weekly
from src.backtest import apply_rule
from src.data import load_snapshot

OUT = config.OUTPUT_DIR / "weekly" / "v1"; OUT.mkdir(parents=True, exist_ok=True)
raw = load_snapshot(); px = raw[["Open", "Close"]]
rf = v2.load_tbill_daily(raw.index)
def proba(path):
    p = pd.read_csv(path, index_col=0, parse_dates=True)
    return p[["p_Sell", "p_Hold", "p_Buy"]].set_axis([-1, 0, 1], axis=1)
P_VAL, P_TEST = proba(config.OUTPUT_DIR / "val_predictions.csv"), proba(config.OUTPUT_DIR / "test_predictions.csv")
def held_for(p, margin):
    return v2.held_from_positions(weekly.weekly_positions_from_actions(apply_rule(p, margin), start=0))

## Signal-rule selection on the 2021–22 weekly validation backtest (cash 0%, as v1)

In [2]:
rows = []
for m in [None, 0.0, 0.05, 0.10, 0.20]:
    h = held_for(P_VAL, m)
    s = v2.summarize(v2.simulate(h, px.loc[h.index]), h, None, "")
    rows.append({"margin": "argmax" if m is None else m, **{k: s[k] for k in ["final_value", "sharpe_excess", "max_drawdown", "round_trips", "exposure", "total_fees"]}})
sel = pd.DataFrame(rows).sort_values(["sharpe_excess", "round_trips"], ascending=[False, True])
bh_h = v2.buy_and_hold_held(P_VAL.index)
bh = v2.summarize(v2.simulate(bh_h, px.loc[P_VAL.index]), bh_h, None, "Buy & Hold")
display(sel)
print("Validation Buy & Hold:", {k: bh[k] for k in ["final_value", "sharpe_excess", "max_drawdown"]})
pick = sel.iloc[0]
MARGIN = None if pick["margin"] == "argmax" else float(pick["margin"])
FROZEN = {"version": "v1w", "model": "RF (E06, unchanged)", "margin": pick["margin"], "schedule": "weekly: decide after the week's last close, fill next open, hold the week",
          "val_sharpe": float(pick["sharpe_excess"]), "val_final_value": float(pick["final_value"]), "val_bh_sharpe": bh["sharpe_excess"],
          "plan_sha256": (config.ROOT / "docs" / "weekly_rebuild_note.sha256").read_text().split()[0], "frozen_before_test": True}
(OUT / "frozen_config.json").write_text(json.dumps(FROZEN, indent=2, default=str))
FROZEN

,margin,final_value,sharpe_excess,max_drawdown,round_trips,exposure,total_fees
0,argmax,116257.66,0.578,-0.1326,20,0.507,4220.04
1,0.0,112135.68,0.464,-0.1326,23,0.429,4701.04
4,0.2,100000.00,0.000,0.0000,0,0.000,0.00
2,0.05,93902.06,-0.115,-0.2166,9,0.507,1733.28
3,0.1,77850.23,-0.683,-0.2566,2,0.489,263.86


Validation Buy & Hold: {'final_value': 104773.32, 'sharpe_excess': 0.217, 'max_drawdown': -0.245}


{'version': 'v1w',
 'model': 'RF (E06, unchanged)',
 'margin': 'argmax',
 'schedule': "weekly: decide after the week's last close, fill next open, hold the week",
 'val_sharpe': 0.578,
 'val_final_value': 116257.66,
 'val_bh_sharpe': 0.217,
 'plan_sha256': '82602bdd44c1797b167933a11fe610c823d14ef2133ef9eed99a742a59ae644a',
 'frozen_before_test': True}

## Test 2023–24 (seen period, see the plan's disclosure)

In [3]:
h = held_for(P_TEST, MARGIN)
pt = px.loc[h.index]; rft = rf.loc[h.index]
res = {}
for cash, r in [("0% (as built)", None), ("T-bills", rft)]:
    sim = v2.simulate(h, pt, r)
    res[cash] = v2.summarize(sim, h, r, f"v1w · cash {cash}")
    bhh = v2.buy_and_hold_held(h.index)
    res[f"Buy & Hold · {cash}"] = v2.summarize(v2.simulate(bhh, pt, r), bhh, r, f"Buy & Hold · cash {cash}")
display(pd.DataFrame(res.values()).set_index("strategy"))
rand = v2.random_timing_benchmark(h, pt, rft, n=1000)
EVID = {"beats_random": round(float((rand < res["T-bills"]["final_value"]).mean()), 3), "random_median_final": float(rand.median())}
attr = v2.out_of_market_attribution(h, px)
print(EVID); display(attr)
(OUT / "results.json").write_text(json.dumps({"frozen": FROZEN, "validation": sel.to_dict(orient="records"), "validation_bh": bh,
    "test": res, "evidence": EVID, "attribution": attr.astype(str).to_dict(orient="records")}, indent=2, default=str))

,final_value,total_return,cagr,ann_vol,sharpe_excess,max_drawdown,round_trips,exposure,total_fees
strategy,,,,,,,,,
v1w · cash 0% (as built),121899.93,0.2190,0.1045,0.0985,1.060,-0.1034,11,0.596,2495.24
Buy & Hold · cash 0% (as built),156546.25,0.5655,0.2523,0.1283,1.820,-0.0997,1,1.000,99.90
v1w · cash T-bills,126864.16,0.2686,0.1269,0.0984,0.756,-0.0970,11,0.596,2554.21
Buy & Hold · cash T-bills,156546.25,0.5655,0.2523,0.1283,1.430,-0.0997,1,1.000,99.90


{'beats_random': 0.269, 'random_median_final': 133028.42688013797}


,out_from,back_in,days,spy_move
0,2023-02-21,2023-03-27,24,-0.0085
1,2023-05-08,2023-06-26,33,0.0515
2,2023-07-10,2023-08-07,20,0.0240
3,2023-08-28,2023-09-18,14,0.0054
4,2024-03-18,2024-04-15,19,0.0022
5,2024-04-29,2024-05-06,5,0.0072
6,2024-06-03,2024-07-22,33,0.0487
7,2024-07-29,2024-08-19,15,0.0160
8,2024-09-09,2024-09-16,5,0.0314
9,2024-11-18,2024-12-23,24,0.0114


4523